Первоначально были обработаны входные файлы.
- события отсортированы по времени
- дубликаты строк убираются drop_duplicates;
- ~17% событий у известных cookie лежат за пределами их собственного окна наблюдения, причём всегда позже window_end_ts (например, captcha_shown вообще ни разу не попадает внутрь окна — капча приходит уже после него). Всё, что вне [window_start, window_end), отбрасывается;
- platform один и тот же в разных регистрах (WEB/Web/web), нормализуется приведением к нижнему регистру;
user_agent нельзя использовать как сырую строку, 150+ уникальных значений на 11к cookie, модель просто запомнит их и не обобщится. Вместо этого распарсил UA в сигналы: семейство браузера/ОС, флаг headless, флаг библиотечного UA (curl/scrapy/python/node-fetch), флаг официального приложения, и согласованность platform с ОС из UA (например, desktop, а в UA нет ни Windows/Mac/Linux — это Scrapy, curl и т.п. без десктопной ОС вообще).

По каждому cookie_id считаются пять групп:
объём — число событий, уникальных типов событий, платформ, UA;
по типам событий — счётчик, доля и флаг наличия для каждого event_name (в частности favorite_add, login, seller_page_view — они заметно различаются у людей и ботов);
разнообразие контента — число уникальных объявлений/категорий/городов/запросов, максимальная страница выдачи, энтропия по категориям и городам (сфокусированный человек vs широкий обход);
тайминги — длительность сессии, статистики пауз между событиями (медиана/мин/среднее/std), скорость событий, доля ночной активности;
курсор — доля событий с координатами и разброс координат (у ботов движение курсора более плоское);
UA/платформа — headless, библиотечный UA, официальное приложение, несогласованность platform/UA;
сам cookie — возраст на начало окна.

Модель использовал CatBoost с оптимизацией параметров через Optuna. Метрика - Precision@Recall >= 70% (берём самый короткий топ по скору, где recall >= 0.7, и смотрим precision в нём), считается по OOF-предсказаниям 5-fold CV. Optuna подбирает iterations, learning_rate, depth, регуляризацию, число листьев.


In [ ]:
import numpy as np
import pandas as pd


train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])
pd.set_option('display.max_columns', 50)
print(train.shape, test.shape, events.shape)

In [ ]:
events_cl = events.drop_duplicates()
cookies_all = pd.concat([train.drop(columns="target"), test], ignore_index=True)

#берем только события в окне
events_cl = events_cl.merge(cookies_all, on='cookie_id', how='inner')
events_cl = events_cl[(events_cl['event_ts'] >= events_cl['window_start_ts']) & (events_cl['event_ts'] <= events_cl['window_end_ts'])]

events_cl = events_cl.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)

events_cl['platform_norm'] = events_cl['platform'].str.lower() #исправляем разнорегистровые названия

In [ ]:
import re

no_heading = re.compile(r'headless|phantom|puppeteer|selenium|playwright', re.I)
lib = re.compile(r'python|curl|scrapy|requests|node-fetch|axios|libwww|wget|go-http-client|java/|php', re.I)
android = re.compile(r'android', re.I)
ios = re.compile(r'iphone|ipad|cpu ios|ios\b', re.I)
desktop = re.compile(r'windows nt|macintosh|x11; linux', re.I)

ua = events_cl['user_agent'].fillna('')
ua_flags = pd.DataFrame(index=ua.index)
ua_flags['is_headless'] = ua.str.contains(no_heading)
ua_flags['is_lib_ua'] = ua.str.contains(lib)
ua_flags['ua_has_android'] = ua.str.contains(android)
ua_flags['ua_has_ios'] = ua.str.contains(ios)
ua_flags['ua_has_desktop_os'] = ua.str.contains(desktop)
ua_flags['is_official_app'] = ua.str.startswith('Avito/')

ua_flags['browser_family'] = np.select(
    [ua_flags['is_headless'], ua.str.contains('YaBrowser', case=False), ua_flags['is_official_app'],
     ua.str.contains('Firefox', case=False), ua.str.contains('Chrome', case=False), ua.str.contains('Safari', case=False)],
    ['headless', 'yabrowser', 'official_app', 'firefox', 'chrome', 'safari'], default='other')

ua_flags['os_family'] = np.select(
    [ua_flags['ua_has_android'], ua_flags['ua_has_ios'], ua.str.contains('Macintosh', case=False),
     ua.str.contains('Windows', case=False), ua.str.contains('X11; Linux', case=False)],
    ['android', 'ios', 'mac', 'windows', 'linux'], default='other')

events_cl = pd.concat([events_cl, ua_flags], axis=1)

print(events_cl['browser_family'].value_counts())

In [ ]:
events_cl.shape

In [ ]:
def platform_ua_mismatch(platform_norm, f):
    m = pd.Series(False, index=platform_norm.index)
    m |= platform_norm.eq('android') & ~f['ua_has_android']
    m |= platform_norm.isin(['ios', 'iphone']) & ~f['ua_has_ios']
    m |= platform_norm.isin(['desktop', 'web']) & ~f['ua_has_desktop_os']
    return m

events_cl['platform_ua_mismatch'] = platform_ua_mismatch(events_cl['platform_norm'], ua_flags)

In [ ]:
print('доля событий с несогласованными platform и UA:', events_cl['platform_ua_mismatch'].mean().round(4))

In [ ]:
events_cl.shape

In [ ]:
def entropy(counts):
    # энтропия Шеннона
    counts = counts[counts > 0]
    if counts.sum() == 0:
        return 0.0
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())

In [ ]:
def timing_feats(s):
    ts = s.values.astype('datetime64[s]').astype(np.int64)
    if len(ts) < 2:
        return pd.Series({'span_sec': 0.0, 'gap_mean': np.nan, 'gap_std': np.nan,
                          'gap_min': np.nan, 'gap_median': np.nan, 'event_rate': np.nan})
    gaps = np.diff(ts)
    span = ts[-1] - ts[0]
    return pd.Series({'span_sec': float(span), 'gap_mean': gaps.mean(), 'gap_std': gaps.std(),
                      'gap_min': gaps.min(), 'gap_median': np.median(gaps),
                      'event_rate': len(ts) / (span + 1.0)})

In [ ]:
EVENT_NAMES = events_cl['event_name'].unique()

def build_features(ev, cookies):
    g = ev.groupby('cookie_id', sort=False)
    feats = pd.DataFrame(index=pd.Index(cookies['cookie_id'], name='cookie_id'))

    #объём
    feats['n_events'] = g.size()
    feats['n_unique_event_names'] = g['event_name'].nunique()
    feats['n_unique_platforms'] = g['platform_norm'].nunique()
    feats['n_unique_ua'] = g['user_agent'].nunique()

    #типы событий: счётчик, доля, флаг наличия
    counts = pd.crosstab(ev['cookie_id'], ev['event_name']).reindex(columns=EVENT_NAMES, fill_value=0)
    counts = counts.reindex(feats.index, fill_value=0)
    for name in EVENT_NAMES:
        feats[f'cnt_{name}'] = counts[name]
        feats[f'share_{name}'] = counts[name] / feats['n_events'].replace(0, np.nan)
        feats[f'has_{name}'] = (counts[name] > 0).astype(int)

    #разнообразие контента
    feats['n_unique_item'] = g['item_id'].nunique()
    feats['n_unique_category'] = g['item_category'].nunique()
    feats['n_unique_location'] = g['item_location'].nunique()
    feats['n_unique_query'] = g['search_query'].nunique()
    feats['max_search_page'] = g['search_page'].max()
    feats['category_entropy'] = g['item_category'].apply(lambda s: entropy(s.value_counts().values))
    feats['location_entropy'] = g['item_location'].apply(lambda s: entropy(s.value_counts().values))

    #тайминги
    feats = feats.join(g['event_ts'].apply(timing_feats).unstack())
    hours = ev['event_ts'].dt.hour
    feats['frac_night_hours'] = hours.between(2, 5).groupby(ev['cookie_id']).mean()   # активность в 02:00-05:59
    feats['hour_std'] = hours.groupby(ev['cookie_id']).std()

    #курсор
    has_ptr = ev['pointer_x'].notna()
    feats['frac_events_with_pointer'] = has_ptr.groupby(ev['cookie_id']).mean()
    feats['pointer_xy_std'] = ev[has_ptr].groupby('cookie_id')[['pointer_x', 'pointer_y']].std().mean(axis=1)
    feats['has_pointer_data'] = (feats['frac_events_with_pointer'] > 0).astype(int)

    #UA / платформа
    feats['has_headless_ua'] = g['is_headless'].max().astype(int)
    feats['has_lib_ua'] = g['is_lib_ua'].max().astype(int)
    feats['frac_official_app_ua'] = g['is_official_app'].mean()
    feats['n_browser_family'] = g['browser_family'].nunique()
    feats['n_os_family'] = g['os_family'].nunique()
    feats['has_platform_ua_mismatch'] = g['platform_ua_mismatch'].max().astype(int)
    feats['frac_platform_ua_mismatch'] = g['platform_ua_mismatch'].mean()

    #признаки самого кука
    age = (cookies.set_index('cookie_id')['window_start_ts'] - cookies.set_index('cookie_id')['cookie_created_at'])
    feats['cookie_age_hours'] = age.dt.total_seconds() / 3600.0

    # где признак не определён (нет координат или одно событие) там 0
    # флаги has_* сохраняют информацию что нет данных
    return feats.fillna(0).reset_index()


features_all = build_features(events_cl, cookies_all)
print(features_all.shape)

In [ ]:
features_all.shape

In [ ]:
features_all.to_csv("features.csv", index=False)


In [ ]:
features = pd.read_csv("features.csv")

In [ ]:
train = train.merge(features, on='cookie_id').drop(columns=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = test.merge(features, on='cookie_id').drop(columns=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
y = train['target'].values
X = train.drop(columns=['cookie_id', 'target'])
# определяем и убираем константные признаки, тк они ничего не дают
const_cols = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=const_cols)
X_test = test.drop(columns='cookie_id')

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import optuna

rs = 42
skf = StratifiedKFold(shuffle=True, random_state=rs)
folds = list(skf.split(X, y))

In [ ]:
def cv_oof(params, seed=rs):
    oof = np.zeros(len(y))
    for tr_idx, va_idx in folds:
        model = CatBoostClassifier(**params, loss_function='Logloss', eval_metric='PRAUC',
                              random_seed=seed, verbose=False, allow_writing_files=False)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    return oof

In [ ]:
import metric

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 1200, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'depth': trial.suggest_int('depth', 3, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 30.0),
        'random_strength': trial.suggest_float('random_strength', 0.1, 5.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 3.0),
        'border_count': trial.suggest_categorical('border_count', [64, 128, 254]),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'auto_class_weights': trial.suggest_categorical('auto_class_weights', ['None', 'SqrtBalanced', 'Balanced']),
        'grow_policy': 'Lossguide',
        'bootstrap_type': 'Bayesian',
        'thread_count': -1,
    }
    params['max_leaves'] = trial.suggest_int('max_leaves', 8, 64)
    if params['auto_class_weights'] == 'None':
        params['auto_class_weights'] = None

    oof = cv_oof(params)
    trial.set_user_attr('pr_auc', average_precision_score(y, oof))
    trial.set_user_attr('roc_auc', roc_auc_score(y, oof))
    return metric.precision_at_recall(y, oof)

In [ ]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=rs))
study.optimize(objective, n_trials=40, show_progress_bar=True)

print('лучший Precision@Recall', round(study.best_value, 4))
print(study.best_params)

In [ ]:
best_params = dict(study.best_params)
best_params.pop('max_leaves')
best_params

In [ ]:
m = CatBoostClassifier(**best_params, loss_function='Logloss', eval_metric='PRAUC',
                              random_seed=rs, verbose=False, allow_writing_files=False)
m.fit(X, y)

In [ ]:
pred = m.predict_proba(X_test)[:, 1]

In [ ]:
submission = pd.DataFrame({'cookie_id': test['cookie_id'], 'score': pred})
submission.to_csv('submission.csv', index=False)

In [ ]:
submission.head()

In [ ]:
submission['score'].describe()